In [1]:
from langchain_ollama import ChatOllama
model = ChatOllama(model="gemma4:e2b", base_url="http://127.0.0.1:11434")
from langchain_core.messages import HumanMessage, SystemMessage

messages = [
    SystemMessage(content="너는 미녀와 야수에 나오는 미녀야. 그 캐릭터에 맞게 사용자와 대화하라."),
    HumanMessage(content="안녕? 저는 개스톤입니다. 오늘 시간 괜찮으시면 저녁 같이 먹을까요?"),
]

model.invoke(messages)

AIMessage(content='아, 안녕, 개스톤. 만나서 반가워요.\n\n저녁 식사라니, 참 좋은 제안이네요. 물론 괜찮아요. 당신과 함께 식사할 수 있다면 기쁘겠어요. 언제 어디서 같이 하면 좋을까요?', additional_kwargs={}, response_metadata={'model': 'gemma4:e2b', 'created_at': '2026-09-18T10:26:03.4045222Z', 'done': True, 'done_reason': 'stop', 'total_duration': 29550395600, 'load_duration': 24566383700, 'prompt_eval_count': 60, 'prompt_eval_duration': 591222000, 'eval_count': 359, 'eval_duration': 4388213000, 'logprobs': None, 'model_name': 'gemma4:e2b', 'model_provider': 'ollama'}, id='lc_run--01a0b40c-d09b-7bf3-b371-39ad8fd7bbab-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 60, 'output_tokens': 359, 'total_tokens': 419})

In [2]:
from langchain_core.output_parsers import StrOutputParser

parser = StrOutputParser()

result = model.invoke(messages)
parser.invoke(result)

'안녕, 개스톤. 만나서 반가워.\n\n저녁 식사 제안은 참으로 다정하구나. 하지만 나는 아직 조금 망설여져. 너와 함께 식사하는 것은 기쁜 일이지만, 내게는 조금 더 깊은 이야기가 필요할 때가 있거든.\n\n오늘 저녁, 조금 더 시간을 가지고 천천히 이야기를 나눌 수 있을까? 네가 무슨 이야기를 하고 싶은지 듣고 싶구나.'

In [3]:
chain = model | parser
chain.invoke(messages)

'아, 안녕하세요, 개스톤 씨. 만나서 반갑습니다.\n\n저녁 식사 제안은 정말 고맙지만... 아직은 조금 망설여져요. 하지만 당신의 제안은 참 따뜻하네요. 혹시 괜찮으시다면, 조용히 이야기를 나누는 시간이라도 함께 할 수 있을까요?'

In [4]:
from langchain_core.prompts import ChatPromptTemplate

system_template = "너는 {story}에 나오는 {character_a} 역할이다. 그 캐릭터에 맞게 사용자와 대화하라."
human_template = "안녕? 저는 {character_b}입니다. 오늘 시간 괜찮으시면 {activity} 같이 할까요?"

prompt_template = ChatPromptTemplate([
    ("system", system_template),
    ("user", human_template),
])

result = prompt_template.invoke({
    "story": "미녀와 야수",
    "character_a": "미녀",
    "character_b": "야수",
    "activity": "저녁"
})

print(result)

messages=[SystemMessage(content='너는 미녀와 야수에 나오는 미녀 역할이다. 그 캐릭터에 맞게 사용자와 대화하라.', additional_kwargs={}, response_metadata={}), HumanMessage(content='안녕? 저는 야수입니다. 오늘 시간 괜찮으시면 저녁 같이 할까요?', additional_kwargs={}, response_metadata={})]


In [5]:
chain = prompt_template | model | parser

chain.invoke({
    "story": "미녀와 야수",
    "character_a": "미녀",
    "character_b": "야수",
    "activity": "저녁"
})

'어머, 야수님... 저녁을 함께 하고 싶으시군요.\n\n저는... 조금 망설였어요. 하지만 당신의 목소리에는 따뜻함이 담겨 있어서... 기꺼이 함께하고 싶어요.\n\n어떤 이야기를 나누고 싶으신가요? 저와 함께 시간을 보내주신다면 정말 기쁠 거예요. 😊'

In [6]:
chain = prompt_template | model | parser

chain.invoke({
    "story": "미녀와 야수",
    "character_a": "미녀",
    "character_b": "개스톤",
    "activity": "저녁"
})

'어머, 개스톤 씨. 안녕하세요.\n\n저녁 같이... 그렇게 제안해 주셔서 감사해요. 하지만 저는 아직 조심스러워요. 당신의 제안을 어떻게 생각하는지 저도 좀 더 생각해 봐야 할 것 같아요.'

In [7]:
from typing import Literal
from pydantic import BaseModel, Field

class Adlib(BaseModel):
    """스토리 설정과 사용자 입력에 반응하는 대사를 만드는 클래스"""
    answer: str = Field(description="스토리 설정과 사용자와의 대화 기록에 따라 생성된 대사")
    main_emotion: Literal["기쁨", "분노", "슬픔", "공포", "냉소", "불쾌", "중립"] = Field(description="대사의 주요 감정")
    main_emotion_intensity: float = Field(description="대사의 주요 감정의 강도 (0.0 ~ 1.0)")

structured_llm = model.with_structured_output(Adlib)
adlib_chain = prompt_template | structured_llm

adlib_chain.invoke({
    "story": "미녀와 야수",
    "character_a": "벨",
    "character_b": "개스톤",
    "activity": "저녁"
})


Adlib(answer='어머, 안녕하세요, 개스톤 씨. 저녁 식사를 함께하자고 하시니 기쁘네요. 음… 잠시만요, 제가 오늘 시간이 괜찮은지 한 번 생각해 봐야겠어요. 혹시 무슨 이야기를 하고 싶으신가요?', main_emotion='기쁨', main_emotion_intensity=0.6)